In [ ]:
!pip install geopandas pyogrio

In [ ]:
import os
import shutil
import zipfile
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

zip_file = "wri_data.gdb.zip"
extract_folder = "wri_extract_temp" # 임시 압축 해제 폴더

print("📦 업로드된 파일 압축 해제 중...")
# 기존에 풀려있던 폴더가 있다면 삭제 후 깔끔하게 다시 풀기
if os.path.exists(extract_folder):
    shutil.rmtree(extract_folder)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

# 2. 풀린 폴더 안에서 .gdb 폴더 찾기
gdb_path = None
for root, dirs, files in os.walk(extract_folder):
    for d in dirs:
        if d.endswith('.gdb'):
            gdb_path = os.path.join(root, d)
            break

# 혹시 zip 안에 바로 파일들이 풀렸을 경우를 대비
if not gdb_path:
    # 압축 푼 임시 폴더 자체를 gdb로 간주하고 이름 변경
    gdb_path = "wri_data.gdb"
    if os.path.exists(gdb_path):
        shutil.rmtree(gdb_path)
    os.rename(extract_folder, gdb_path)
    extract_folder = None # 임시 폴더 이름 변경했으므로 None 처리

print(f"✅ GDB 경로 확인 완료: {gdb_path}")

# 3. WRI 데이터 로드 및 분석 시작
if gdb_path and os.path.exists(gdb_path):
    print("⏳ WRI 데이터를 읽어오는 중입니다. 데이터 크기에 따라 시간이 걸릴 수 있습니다...")
    # pyogrio 엔진을 사용해 빠르게 읽어오기
    gdf_wri = gpd.read_file(gdb_path, engine="pyogrio")

    if os.path.exists("위경도.xlsx"):
        # 내 사업장 데이터 로드
        df_my = pd.read_excel("위경도.xlsx")
        df_my = df_my.dropna(subset=['경도(Lon)', '위도(Lat)'])

        # Point 객체 생성 및 GeoDataFrame 변환
        geometry = [Point(xy) for xy in zip(df_my['경도(Lon)'], df_my['위도(Lat)'])]
        gdf_my = gpd.GeoDataFrame(df_my, geometry=geometry, crs="EPSG:4326")

        # WRI 데이터와 좌표계 통일
        gdf_wri = gdf_wri.to_crs(gdf_my.crs)

        print("🔍 좌표 매핑 및 리스크 분석 중...")
        # sjoin(spatial join)으로 위치 기반 데이터 합치기
        result = gpd.sjoin(gdf_my, gdf_wri, how="left", predicate="within")

        # 결과 저장
        result.to_excel("KOSPI50_수자원리스크_완료.xlsx", index=False)
        print("🎉 분석 성공! 'KOSPI50_수자원리스크_완료.xlsx' 파일이 생성되었습니다.")

        # 정리: 임시 폴더 삭제
        if extract_folder and os.path.exists(extract_folder):
             shutil.rmtree(extract_folder)

    else:
        print("❌ '위경도.xlsx' 파일이 없습니다. 파일을 업로드해 주세요.")
else:
    print("❌ GDB 폴더를 제대로 인식하지 못했습니다. 압축 파일 구조를 확인해 주세요.")

📦 업로드된 파일 압축 해제 중...
✅ GDB 경로 확인 완료: wri_data.gdb
⏳ WRI 데이터를 읽어오는 중입니다. 데이터 크기에 따라 시간이 걸릴 수 있습니다...


/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'wri_data.gdb': 'baseline_annual' (default), 'baseline_monthly', 'future_annual'. Specify layer parameter to avoid this warning.
  result = read_func(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


🔍 좌표 매핑 및 리스크 분석 중...
🎉 분석 성공! 'KOSPI50_수자원리스크_완료.xlsx' 파일이 생성되었습니다.
